# 07. Supplementary analysis

Supplementary analyses supporting the main manuscript.

**7 tasks** (AR −1 excluded): HBD +1, HBD −1, HBA +1, HBA −1, RotB +1, RotB −1, AR +1
Evaluation unit: parent × task. `candidate_rank` used as-is; no shuffling.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import ACTIVE_PARENT_DATASET, TASKS, TASK_NAMES, K_VALUES
from utils.evaluation import success_at_k, success_rate_at_k, success_rate_by_task_at_k
from utils.records import compute_first_success_rank


In [ ]:
RESULTS_DIR  = Path("../results/new_benchmark")
SUPP_DIR     = RESULTS_DIR / "supplementary"
SUPP_FIG_DIR = SUPP_DIR / "figures"
SUPP_DIR.mkdir(parents=True, exist_ok=True)
SUPP_FIG_DIR.mkdir(parents=True, exist_ok=True)

PARENT_FILE = Path(ACTIVE_PARENT_DATASET)

# Primary 4 methods
METHOD_FILES = {
    "Scaffold-tuner": RESULTS_DIR / "scaffold_tuner.csv",
    "Random":         RESULTS_DIR / "random_local_editing.csv",
    "CReM":           RESULTS_DIR / "crem.csv",
    "LibInvent (RL)": RESULTS_DIR / "libinvent.csv",
}
METHOD_ORDER = list(METHOD_FILES.keys())

# Supplementary control: prior-only sampling
PRIOR_ONLY_FILE = RESULTS_DIR / "libinvent_prior_only.csv"

parents = pd.read_csv(PARENT_FILE)
if "parent_id" not in parents.columns and "chembl_id" in parents.columns:
    parents = parents.rename(columns={"chembl_id": "parent_id"})
all_parent_ids = parents["parent_id"].unique()

results = {}
for method, path in METHOD_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        if "parent_id" not in df.columns and "chembl_id" in df.columns:
            df = df.rename(columns={"chembl_id": "parent_id"})
        results[method] = df
        print(f"  {method}: {len(df):,} records")
    else:
        print(f"  {method}: NOT FOUND")

methods_present = [m for m in METHOD_ORDER if m in results]
print(f"Parents: {len(all_parent_ids)}")


## Table S1: Generation statistics

In [ ]:
gen_stat_files = {
    "Random":         RESULTS_DIR / "random_generation_stats.csv",
    "CReM":           RESULTS_DIR / "crem_generation_stats.csv",
    "LibInvent (RL)": RESULTS_DIR / "libinvent_generation_stats.csv",
}

gen_rows = []
for method, path in gen_stat_files.items():
    if not path.exists():
        print(f"  {method}: stats file not found"); continue
    gs = pd.read_csv(path)

    # Generation statistics definitions (exclusive categories):
    #   n_examined = n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique
    #
    # Validity         = (n_examined - n_invalid) / n_examined
    #   → fraction of edits that produced an RDKit-valid molecule
    # Scaffold retention = (n_valid - n_scaffold_failed) / n_valid
    #   → fraction of valid molecules that retained the parent scaffold
    #   where n_valid = n_examined - n_invalid
    # Uniqueness       = n_valid_unique / (n_valid_unique + n_duplicate)
    #   → fraction of scaffold-retained valid molecules that are non-duplicate

    n_ex    = gs["n_examined"].sum()
    n_inv   = gs["n_invalid"].sum()
    n_scf   = gs["n_scaffold_failed"].sum()
    n_vu    = gs["n_valid_unique"].sum()
    n_dup   = gs["n_duplicate"].sum()
    n_valid = n_ex - n_inv          # RDKit-valid molecules

    gen_rows.append({
        "method":               method,
        "n_examined":          int(n_ex),
        "validity_%":           round(100 * n_valid / n_ex, 1) if n_ex else None,
        "scaffold_retention_%": round(100 * (n_valid - n_scf) / n_valid, 1) if n_valid else None,
        "uniqueness_%":         round(100 * n_vu / (n_vu + n_dup), 1) if (n_vu + n_dup) else None,
    })

pd.DataFrame(gen_rows).set_index("method")


## Table S2: Task-wise Success@k

In [ ]:
task_rows = []
for method, df in results.items():
    for k in K_VALUES:
        by_task = success_rate_by_task_at_k(df, k, all_parent_ids, "success")
        for task, rate in by_task.items():
            task_rows.append({"method": method, "k": k, "task": task, "success_%": round(rate*100, 1)})

df_task = pd.DataFrame(task_rows)
display(df_task[df_task["k"]==10].pivot(index="task", columns="method", values="success_%").reindex(TASK_NAMES))


## Table S3: Task-wise Strict Success@k

In [ ]:
strict_task_rows = []
for method, df in results.items():
    for k in K_VALUES:
        by_task = success_rate_by_task_at_k(df, k, all_parent_ids, "strict_success")
        for task, rate in by_task.items():
            strict_task_rows.append({"method": method, "k": k, "task": task, "strict_%": round(rate*100, 1)})

df_strict_task = pd.DataFrame(strict_task_rows)
display(df_strict_task[df_strict_task["k"]==10].pivot(index="task", columns="method", values="strict_%").reindex(TASK_NAMES))


## Figure S1: Task-wise Success@k curves

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharey=False)
for ax, task in zip(axes.flatten(), TASK_NAMES):
    for method in methods_present:
        part = df_task[(df_task["method"]==method) & (df_task["task"]==task)].sort_values("k")
        ax.plot(part["k"], part["success_%"], marker="o", linewidth=1.5, label=method)
    ax.set_title(task, fontsize=9)
    ax.set_xscale("log"); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES, fontsize=7)
    ax.set_xlabel("k", fontsize=8); ax.set_ylabel("Success@k (%)", fontsize=8); ax.grid(True, alpha=0.3)

if len(TASK_NAMES) < len(axes.flatten()):
    axes.flatten()[-1].set_visible(False)
axes.flatten()[0].legend(fontsize=7)
fig.suptitle("Task-wise Success@k", fontsize=11)
fig.tight_layout()
fig.savefig(SUPP_FIG_DIR / "figS1_task_success_curves.pdf", dpi=200)
plt.show()


## Figure S2: Collateral descriptor change (successful candidates)

In [ ]:
coll_data = [(m, results[m][results[m]["success"]==True]["collateral_change"].dropna().values)
             for m in methods_present if "collateral_change" in results[m].columns]
if coll_data:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot([v for _,v in coll_data], tick_labels=[m for m,_ in coll_data], showfliers=False)
    ax.set_ylabel("Collateral change (Σ|Δdesc| of non-target)")
    ax.set_title("Collateral descriptor change among successful candidates")
    plt.xticks(rotation=15, ha="right"); fig.tight_layout()
    fig.savefig(SUPP_FIG_DIR / "figS2_collateral_change.pdf", dpi=200)
    plt.show()


## Figure S3: Tanimoto similarity to parent (successful candidates)

In [ ]:
tan_data = [(m, results[m][results[m]["success"]==True]["tanimoto_to_parent"].dropna().values)
            for m in methods_present if "tanimoto_to_parent" in results[m].columns]
if tan_data:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot([v for _,v in tan_data], tick_labels=[m for m,_ in tan_data], showfliers=False)
    ax.set_ylabel("Tanimoto similarity to parent (Morgan fp, r=2)")
    ax.set_title("Structural locality of successful candidates")
    ax.set_ylim(0, 1); plt.xticks(rotation=15, ha="right"); fig.tight_layout()
    fig.savefig(SUPP_FIG_DIR / "figS3_tanimoto_similarity.pdf", dpi=200)
    plt.show()


## Figure S4: Scaffold retention by task

In [ ]:
sc_rows = []
for method, df in results.items():
    for task in TASK_NAMES:
        sub = df[df["task"]==task]
        if len(sub)==0: continue
        rate = sub["scaffold_retained"].mean()*100 if "scaffold_retained" in sub.columns else None
        sc_rows.append({"method": method, "task": task, "scaffold_retention_%": rate})

df_sc = pd.DataFrame(sc_rows)
pivot_sc = df_sc.pivot(index="task", columns="method", values="scaffold_retention_%").reindex(TASK_NAMES)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(TASK_NAMES)); width = 0.8/max(len(methods_present),1)
for i, method in enumerate(methods_present):
    vals = pivot_sc.get(method, pd.Series(dtype=float))
    ax.bar(x+(i-len(methods_present)/2+0.5)*width,
           [vals.get(t,0) for t in TASK_NAMES], width, label=method)
ax.set_xticks(x); ax.set_xticklabels(TASK_NAMES, rotation=30, ha="right")
ax.set_ylim(0, 105); ax.set_ylabel("Scaffold retention (%)"); ax.legend()
fig.tight_layout()
fig.savefig(SUPP_FIG_DIR / "figS4_scaffold_retention.pdf", dpi=200)
plt.show()


## Figure S5: First-success rank ECDF

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method in methods_present:
    df_m = results[method]
    fsr = compute_first_success_rank(df_m, all_parent_ids)
    vals = fsr["first_success_rank"].dropna().sort_values().values
    if len(vals) == 0: continue
    ecdf_x = np.concatenate([[0], vals, [K_VALUES[-1]]])
    ecdf_y = np.concatenate([[0], np.arange(1, len(vals)+1)/len(all_parent_ids)/len(TASK_NAMES),
                              [len(vals)/len(all_parent_ids)/len(TASK_NAMES)]])
    ax.step(ecdf_x, ecdf_y*100, where="post", linewidth=1.8, label=method)

ax.set_xlabel("Candidate rank of first success")
ax.set_ylabel("Cumulative fraction of parent×task pairs (%)")
ax.set_title("Figure S5: ECDF of first-success rank")
ax.set_xlim(0, 100); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(SUPP_FIG_DIR / "figS5_first_success_ecdf.pdf", dpi=200)
plt.show()


## Figure S6: LibInvent RL vs Prior-only (control comparison)

In [ ]:
if PRIOR_ONLY_FILE.exists():
    df_prior = pd.read_csv(PRIOR_ONLY_FILE)
    if "parent_id" not in df_prior.columns and "chembl_id" in df_prior.columns:
        df_prior = df_prior.rename(columns={"chembl_id": "parent_id"})

    fig, ax = plt.subplots(figsize=(7, 5))
    for label, df_cmp in [("LibInvent (RL)", results.get("LibInvent (RL)")),
                           ("LibInvent (prior-only)", df_prior)]:
        if df_cmp is None: continue
        s_vals = [success_rate_at_k(df_cmp, k, all_parent_ids)*100 for k in K_VALUES]
        ax.plot(K_VALUES, s_vals, marker="o", linewidth=1.8, label=label)

    ax.set_xscale("log"); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
    ax.set_xlabel("k"); ax.set_ylabel("Success rate (%)")
    ax.set_title("Figure S6: LibInvent RL vs Prior-only sampling")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(SUPP_FIG_DIR / "figS6_libinvent_rl_vs_prior.pdf", dpi=200)
    plt.show()
else:
    print("Prior-only results not found. Run 05b_reinvent4_libinvent_prior_sampling.ipynb first.")


## Table S4: Scaffold-tuner Tier 1 / Tier 2 breakdown

In [ ]:
if "Scaffold-tuner" in results and "generation_strategy" in results["Scaffold-tuner"].columns:
    df_st = results["Scaffold-tuner"]
    tier_cnt = df_st.groupby(["task","generation_strategy"]).size().unstack(fill_value=0)
    tier_suc = df_st[df_st["success"]==True].groupby(["task","generation_strategy"]).size().unstack(fill_value=0)
    print("Candidate count by Tier:"); display(tier_cnt.reindex(TASK_NAMES))
    print("\nSuccess count by Tier:"); display(tier_suc.reindex(TASK_NAMES))
else:
    print("generation_strategy column not available.")
